# Reproduce: gap-protected spectral invariants on T^d/Z_2

This notebook recomputes the headline results of the paper from first
principles and then runs the full verification suite. Run all cells
top to bottom. The setup cell makes it work in Google Colab or any fresh
Jupyter (it clones the repository and installs the dependencies when they
are not already present); locally it is a no-op.

Paper: *Gap-protected spectral invariants on T^d/Z_2: dimensional rigidity
at d = 4* (doi:10.5281/zenodo.20597196).


In [ ]:
# Setup: no-op locally; in Colab/fresh Jupyter it fetches the repo and deps.
import os, subprocess, sys
if not os.path.exists('run_all.py'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/kootru-repo/'
                    'gap-protected-spectral-invariants', '_repo'], check=True)
    os.chdir('_repo')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', 'requirements.txt'], check=True)
print('ready in', os.getcwd())


In [ ]:
import itertools, math
import mpmath as mp
mp.mp.dps = 30

# --- mode counts on the Z^4 covering torus ---
def N4(K):
    return sum(1 for n in itertools.product(range(-3, 4), repeat=4)
               if sum(x*x for x in n) <= K)
r4 = [N4(k) - (N4(k-1) if k else 0) for k in range(6)]
print('shell counts r_4(k), k=0..5 :', r4)
print('N_4(5) =', N4(5), '= 1 + 8 + 128 =', 1 + 8 + 128,
      '  (b_0 + chi_orb + |Fix|*chi_orb)')

# --- smooth spectral action (alpha-free closed form) ---
pi, gE = mp.pi, mp.euler
F0 = 1 - mp.log(pi) + 6*mp.zeta(2, 1, 1)/pi**2 + mp.log(4)/3
D1 = F0 + gE/2
print('F_0                         =', mp.nstr(F0, 10))
print('Delta_1 = F_0 + gamma_E/2   =', mp.nstr(D1, 10))
print('smooth action 137 + Delta_1 =', mp.nstr(137 + D1, 13))

# --- scattering radius rho from the exact theta identities ---
ln2 = mp.log(2)
Z = [-8*ln2, (pi - 2*ln2)/2, -2*ln2, (-pi - 2*ln2)/2, -4*ln2]  # Z^(h)(1)
G = [z/(4*pi**2) for z in Z]
def kraw(h, w, d=4):
    return sum((-1)**j * math.comb(w, j) * math.comb(d - w, h - j)
               for j in range(h + 1) if 0 <= h - j <= d - w)
mu = [sum(G[h]*kraw(h, w) for h in range(5)) for w in range(5)]
rho = max(abs(m) for m in mu)/(8*pi**2)
print('mu_w  =', [mp.nstr(m, 4) for m in mu])
print('rho   =', mp.nstr(rho, 4), '  (< 7.2e-3)')
lo, hi = 137 + D1 - rho**2/(1 - rho), 137 + D1 + rho**2/(1 - rho)
print('Born interval = [%s, %s]' % (mp.nstr(lo, 9), mp.nstr(hi, 9)))


## Full verification suite

The cell below runs every script in the repository (the same command the
continuous-integration workflow runs). It exits cleanly only if all checks
pass.


In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, 'run_all.py'], capture_output=True, text=True)
print(r.stdout[-2000:])
print('exit code:', r.returncode)
